# Regresión Logística — Tres Modelos por Régimen Educativo
## Predicción de Deserción Estudiantil
**Equipo 7 · Desarrollo de Aplicaciones Avanzadas de Ciencias Computacionales**

---
Entrena tres regresiones logísticas y compara sus métricas (Precision, Recall, F1) y valores SHAP:

| Modelo | Datos de entrenamiento |
|--------|------------------------|
| **Todos** | Todos los alumnos (AD14–AD20) |
| **PreTec21** | Solo alumnos con `educational.model == 0` |
| **Tec21** | Solo alumnos con `educational.model == 1` |

## Instalación de dependencias

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec('shap') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
print('shap disponible:', importlib.util.find_spec('shap') is not None)

## Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, time
from pathlib import Path

import shap

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
print('Librerías cargadas correctamente.')
print(f'  shap     : {shap.__version__}')
print(f'  sklearn  : {__import__("sklearn").__version__}')
print(f'  pandas   : {pd.__version__}')
print(f'  numpy    : {np.__version__}')

## Variables globales

In [ ]:
SEED       = 42
TARGET     = 'retention'
N_SPLITS   = 5
DATA_DIR   = Path('../data')

np.random.seed(SEED)

## Carga de datos
La primera ejecución lee el `.xlsx` (~30 s) y guarda un CSV caché. Las siguientes leen el CSV (~3 s).

In [ ]:
CACHE_CSV = DATA_DIR / 'dataset_cache.csv'
XLSX_PATH = DATA_DIR / 'dataset.xlsx'

if CACHE_CSV.exists():
    t0 = time.time()
    df_raw = pd.read_csv(CACHE_CSV, low_memory=False)
    print(f'✓ Cache CSV cargado en {time.time()-t0:.1f}s  →  {df_raw.shape}')
else:
    print('Primera carga desde xlsx (puede tardar ~30s)...')
    t0 = time.time()
    df_raw = pd.read_excel(XLSX_PATH, engine='openpyxl')
    df_raw.to_csv(CACHE_CSV, index=False)
    print(f'✓ Xlsx leído y cache guardado en {time.time()-t0:.1f}s  →  {df_raw.shape}')

## Preprocesamiento
Mismo pipeline que `exploration.ipynb`: filtrado, exclusión de columnas con fuga, imputación MCAR/MAR/MNAR, ingeniería de variables y encoding.

In [ ]:
# ── 1. Filtrar universitarios ────────────────────────────────────────────────
df = df_raw[df_raw['level'] == 'Undergraduate'].copy()
print(f'Universitarios: {len(df):,}  |  PreTec21: {(df["educational.model"]==0).sum():,}  |  Tec21: {(df["educational.model"]==1).sum():,}')

In [ ]:
# ── 2. Eliminar columnas con fuga o irrelevantes ─────────────────────────────
DROP_COLS = [
    'student.id', 'level',
    'average.first.period', 'failed.subject.first.period', 'dropped.subject.first.period',
    'dropout.semester', 'program', 'id.school.origin',
    'scholarship.type', 'school.cost', 'general.math.eval',
    'parents.exatec', 'father.exatec', 'mother.exatec',
    'father.education.complete', 'father.education.summary',
    'mother.education.complete', 'mother.education.summary',
    'scholarship.perc', 'loan.perc',
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
print(f'Columnas tras exclusión: {df.shape[1]}')

In [ ]:
# ── 3. MCAR: normalizar admission.test ──────────────────────────────────────
def norm_admission_test(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.lower().startswith('does not'): return np.nan
    try:
        f = float(s)
        return max(0.0, (f - 400) / 1200.0) if f > 100 else f / 100.0
    except Exception:
        return np.nan

df['admission_test_norm'] = df['admission.test'].apply(norm_admission_test)
df.drop(columns=['admission.test'], inplace=True)

med_at = df['admission_test_norm'].median()
med_ar = df['admission.rubric'].median()
df['admission_test_norm'].fillna(med_at, inplace=True)
df['admission.rubric'].fillna(med_ar, inplace=True)
print(f'admission_test_norm imputado con mediana: {med_at:.3f}')
print(f'admission.rubric    imputado con mediana: {med_ar:.2f}')

In [ ]:
# ── 4. MAR: bandera has_extracurriculars ─────────────────────────────────────
ACTIVITY_COLS = [
    'physical.education', 'cultural.diffusion', 'student.society',
    'total.life.activities', 'athletic.sports', 'art.culture',
    'student.society.leadership', 'life.work.mentoring', 'wellness.activities'
]

def parse_activity(val):
    if pd.isna(val): return 0
    s = str(val).strip()
    if s in ('Does not apply', 'No information', '', '0', '0.0'): return 0
    try: return int(float(s) > 0)
    except Exception: return 0

df['has_extracurriculars'] = df['total.life.activities'].apply(parse_activity)
df.drop(columns=[c for c in ACTIVITY_COLS if c in df.columns], inplace=True)
print(f'has_extracurriculars — con actividades: {df["has_extracurriculars"].sum():,} ({df["has_extracurriculars"].mean()*100:.1f}%)')

In [ ]:
# ── 5. MNAR: first.generation → 3 categorías ────────────────────────────────
def encode_first_gen(val):
    s = str(val).strip()
    if s == 'Yes': return 1
    if s == 'No':  return 0
    return -1

df['first_gen_enc'] = df['first.generation'].apply(encode_first_gen)
df.drop(columns=['first.generation'], inplace=True)

In [ ]:
# ── 6. Ingeniería y encoding de categóricas ──────────────────────────────────
EDU_ORD = {'No information': 0, 'No degree': 1, 'Undergraduate degree': 2, 'Master degree': 3, 'PhD': 4}
df['educ_padres_max'] = df['max.degree.parents'].map(EDU_ORD).fillna(0).astype(int)
df.drop(columns=['max.degree.parents'], inplace=True)

df.rename(columns={'total.scholarship.loan': 'apoyo_financiero'}, inplace=True)

df['gender_enc']     = df['gender'].map({'Male': 1, 'Female': 0}).fillna(0).astype(int)
df['tec_enc']        = df['tec.no.tec'].map({'TEC': 1, 'NO TEC': 0}).fillna(0).astype(int)
df['foreign_enc']    = df['foreign'].map({'Local': 0, 'Yes: National': 1, 'Yes: Foreigner': 2}).fillna(0).astype(int)
df['zone_enc']       = df['zone.type'].map({'Rural': 0, 'Semiurban': 1, 'Urban': 2, 'No information': 1}).fillna(1).astype(int)

sec_map = {'No information': 0, 'Level 1': 1, 'Level 2': 2, 'Level 3': 3,
           'Level 4': 4, 'Level 5': 5, 'Level 6': 6, 'Level 7': 7}
df['socioec_enc']    = df['socioeconomic.level'].map(sec_map).fillna(0).astype(int)

lag_map = {'No information': 0, 'Low': 1, 'Medium': 2, 'High': 3}
df['social_lag_enc'] = df['social.lag'].map(lag_map).fillna(0).astype(int)

le_region = LabelEncoder()
df['region_enc']     = le_region.fit_transform(df['region'].fillna('Unknown'))

le_school = LabelEncoder()
df['school_enc']     = le_school.fit_transform(df['school'].fillna('Unknown'))

df.drop(columns=['gender', 'tec.no.tec', 'foreign', 'zone.type',
                 'socioeconomic.level', 'social.lag', 'region', 'school'], inplace=True)

print('Preprocesamiento completo.')
print(f'Shape final: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')

In [ ]:
# ── 7. Definir columnas de features ─────────────────────────────────────────
FEATURE_COLS = [
    'PNA', 'admission_test_norm', 'english.evaluation', 'admission.rubric',
    'FTE', 'apoyo_financiero', 'has_extracurriculars',
    'first_gen_enc', 'educ_padres_max',
    'age', 'gender_enc', 'tec_enc', 'foreign_enc',
    'zone_enc', 'socioec_enc', 'social_lag_enc',
    'region_enc', 'school_enc', 'online.test'
]

print(f'Features del modelo: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

## Definición de los tres subconjuntos de entrenamiento

| Modelo | Filtro | Estrategia de evaluación |
|--------|--------|--------------------------|
| **Todos** | Todos los alumnos universitarios | Stratified 5-Fold CV |
| **PreTec21** | `educational.model == 0` | Stratified 5-Fold CV |
| **Tec21** | `educational.model == 1` | Stratified 5-Fold CV |

In [ ]:
mask_all     = pd.Series([True]  * len(df), index=df.index)
mask_pretec  = df['educational.model'] == 0
mask_tec21   = df['educational.model'] == 1

subsets = {
    'Todos'   : mask_all,
    'PreTec21': mask_pretec,
    'Tec21'   : mask_tec21,
}

for name, mask in subsets.items():
    sub = df.loc[mask]
    n_dropout = (sub[TARGET] == 0).sum()
    print(f'{name:10s}: {mask.sum():>6,} alumnos  |  desertores: {n_dropout:,} ({n_dropout/mask.sum()*100:.1f}%)')

## Función auxiliar — Entrenar y evaluar

In [ ]:
def train_and_evaluate(X, y, model_name, feature_names, n_splits=5, seed=42):
    """
    Entrena una Regresión Logística con Stratified K-Fold CV.
    Devuelve el modelo final entrenado en todos los datos y las métricas CV.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # Pipeline: imputación de seguridad + escalado + LR
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('lr',      LogisticRegression(
            class_weight='balanced', max_iter=2000,
            random_state=seed, solver='lbfgs'
        )),
    ])

    print(f'\n{'='*60}')
    print(f'  Modelo: {model_name}')
    print(f'  N total: {len(y):,}  |  Desertores: {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)')
    print(f'  K-Fold: {n_splits} pliegues estratificados')
    print(f'{'='*60}')

    cv_results = cross_validate(
        pipeline, X, y,
        cv=skf,
        scoring=['precision_weighted', 'recall_weighted', 'f1_weighted'],
        n_jobs=-1
    )

    prec  = cv_results['test_precision_weighted']
    rec   = cv_results['test_recall_weighted']
    f1    = cv_results['test_f1_weighted']

    print(f'\n  Métricas CV (media ± std sobre {n_splits} pliegues):')
    print(f'  Precision : {prec.mean():.4f} ± {prec.std():.4f}')
    print(f'  Recall    : {rec.mean():.4f} ± {rec.std():.4f}')
    print(f'  F1-Score  : {f1.mean():.4f} ± {f1.std():.4f}')

    # Modelo final entrenado en todos los datos del subconjunto
    pipeline.fit(X, y)
    print(f'\n  Modelo final entrenado en {len(y):,} muestras.')

    metrics = {
        'precision_mean': prec.mean(), 'precision_std': prec.std(),
        'recall_mean'   : rec.mean(),  'recall_std'   : rec.std(),
        'f1_mean'       : f1.mean(),   'f1_std'       : f1.std(),
    }
    return pipeline, metrics

## Función auxiliar — SHAP values

In [ ]:
def plot_shap(pipeline, X_raw, feature_names, model_name, max_display=15):
    """
    Calcula y grafica los SHAP values usando LinearExplainer.
    Transforma X_raw con el pipeline (imputer + scaler) antes de pasar al explainer.
    """
    # Transformar datos con los pasos previos al LR
    imputer = pipeline.named_steps['imputer']
    scaler  = pipeline.named_steps['scaler']
    lr      = pipeline.named_steps['lr']

    X_transformed = scaler.transform(imputer.transform(X_raw))

    # Submuestra de hasta 2000 filas para eficiencia
    rng = np.random.default_rng(42)
    n_sample = min(2000, X_transformed.shape[0])
    idx = rng.choice(X_transformed.shape[0], size=n_sample, replace=False)
    X_sample = X_transformed[idx]

    explainer   = shap.LinearExplainer(lr, X_transformed)
    shap_values = explainer.shap_values(X_sample)

    # Si shap_values es lista (multiclass), tomar la clase 1 (retención)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    print(f'\n  SHAP Values — {model_name} (muestra: {n_sample:,} alumnos)')
    print(f'  Shape de shap_values: {shap_values.shape}')

    # Beeswarm plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values, X_sample,
        feature_names=feature_names,
        max_display=max_display,
        show=False
    )
    plt.title(f'SHAP Values — {model_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Bar plot (importancia media absoluta)
    plt.figure(figsize=(9, 5))
    shap.summary_plot(
        shap_values, X_sample,
        feature_names=feature_names,
        plot_type='bar',
        max_display=max_display,
        show=False
    )
    plt.title(f'SHAP — Importancia media |SHAP| — {model_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Tabla de importancia media
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame({'Feature': feature_names, 'mean_|SHAP|': mean_abs_shap})
    shap_df = shap_df.sort_values('mean_|SHAP|', ascending=False).reset_index(drop=True)
    print(f'\n  Top {max_display} variables por |SHAP| medio — {model_name}:')
    print(shap_df.head(max_display).to_string(index=False))

    return shap_df

---
## Modelo 1 — Todos los alumnos

In [ ]:
mask = mask_all
X_all = df.loc[mask, FEATURE_COLS].values
y_all = df.loc[mask, TARGET].values

pipeline_all, metrics_all = train_and_evaluate(
    X_all, y_all,
    model_name='Todos los alumnos',
    feature_names=FEATURE_COLS
)

In [ ]:
shap_df_all = plot_shap(
    pipeline_all, X_all, FEATURE_COLS,
    model_name='Todos los alumnos'
)

---
## Modelo 2 — PreTec21 (`educational.model == 0`)

In [ ]:
mask = mask_pretec
X_pretec = df.loc[mask, FEATURE_COLS].values
y_pretec = df.loc[mask, TARGET].values

pipeline_pretec, metrics_pretec = train_and_evaluate(
    X_pretec, y_pretec,
    model_name='PreTec21 (educational.model == 0)',
    feature_names=FEATURE_COLS
)

In [ ]:
shap_df_pretec = plot_shap(
    pipeline_pretec, X_pretec, FEATURE_COLS,
    model_name='PreTec21'
)

---
## Modelo 3 — Tec21 (`educational.model == 1`)

In [ ]:
mask = mask_tec21
X_tec21 = df.loc[mask, FEATURE_COLS].values
y_tec21 = df.loc[mask, TARGET].values

pipeline_tec21, metrics_tec21 = train_and_evaluate(
    X_tec21, y_tec21,
    model_name='Tec21 (educational.model == 1)',
    feature_names=FEATURE_COLS
)

In [ ]:
shap_df_tec21 = plot_shap(
    pipeline_tec21, X_tec21, FEATURE_COLS,
    model_name='Tec21'
)

---
## Comparación de los tres modelos

In [ ]:
comparison = pd.DataFrame({
    'Modelo'    : ['Todos', 'PreTec21', 'Tec21'],
    'N'         : [len(y_all), len(y_pretec), len(y_tec21)],
    'Precision' : [f"{metrics_all['precision_mean']:.4f} ± {metrics_all['precision_std']:.4f}",
                   f"{metrics_pretec['precision_mean']:.4f} ± {metrics_pretec['precision_std']:.4f}",
                   f"{metrics_tec21['precision_mean']:.4f} ± {metrics_tec21['precision_std']:.4f}"],
    'Recall'    : [f"{metrics_all['recall_mean']:.4f} ± {metrics_all['recall_std']:.4f}",
                   f"{metrics_pretec['recall_mean']:.4f} ± {metrics_pretec['recall_std']:.4f}",
                   f"{metrics_tec21['recall_mean']:.4f} ± {metrics_tec21['recall_std']:.4f}"],
    'F1-Score'  : [f"{metrics_all['f1_mean']:.4f} ± {metrics_all['f1_std']:.4f}",
                   f"{metrics_pretec['f1_mean']:.4f} ± {metrics_pretec['f1_std']:.4f}",
                   f"{metrics_tec21['f1_mean']:.4f} ± {metrics_tec21['f1_std']:.4f}"],
})

print('\n' + '='*70)
print('  RESUMEN COMPARATIVO — Regresión Logística')
print('  (Stratified 5-Fold CV · weighted avg · class_weight=balanced)')
print('='*70)
print(comparison.to_string(index=False))

In [ ]:
# ── Gráfica comparativa de F1, Precision y Recall ───────────────────────────
modelos   = ['Todos', 'PreTec21', 'Tec21']
all_m     = [metrics_all,  metrics_pretec, metrics_tec21]

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
metric_pairs = [
    ('precision_mean', 'precision_std', 'Precision', '#2563eb'),
    ('recall_mean',    'recall_std',    'Recall',    '#059669'),
    ('f1_mean',        'f1_std',        'F1-Score',  '#d97706'),
]

for ax, (mean_key, std_key, label, color) in zip(axes, metric_pairs):
    means = [m[mean_key] for m in all_m]
    stds  = [m[std_key]  for m in all_m]
    bars  = ax.bar(modelos, means, yerr=stds, color=color, alpha=0.85,
                   capsize=6, edgecolor='white')
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Comparación de métricas por modelo (CV 5-Fold, weighted)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 10 variables SHAP más importantes por modelo ────────────────────────
print('\nTop 10 variables por |SHAP| medio — comparación entre modelos:')
print('\n  Todos:')
print(shap_df_all.head(10)[['Feature','mean_|SHAP|']].to_string(index=False))
print('\n  PreTec21:')
print(shap_df_pretec.head(10)[['Feature','mean_|SHAP|']].to_string(index=False))
print('\n  Tec21:')
print(shap_df_tec21.head(10)[['Feature','mean_|SHAP|']].to_string(index=False))